# Análise de um Escritório de Contabilidade
**Disciplina:** Linguagem de Programação — Análise e Visualização de Dados com Python  
**Avaliação:** G1  
**Aluno:** Yago Baltazar Ferreira  
**Professor:** Alexandre Neves Louzada

## 1. Introdução ao problema
Um escritório de contabilidade precisa acompanhar sua carteira de clientes, o faturamento apurado, os impostos de cada regime tributário (Simples Nacional, Lucro Presumido e Lucro Real), a receita de honorários e a inadimplência.

**Perguntas de negócio**
- Qual a carga tributária média de cada regime?
- Como o faturamento e os impostos evoluem ao longo do tempo?
- Quais estados e setores concentram mais receita?
- Qual a inadimplência de honorários e onde ela é maior?
- O faturamento se relaciona com impostos e honorários?

## 2. Explicação da base de dados
Os dados são **fictícios** (gerados com `gerar_dados.py`, semente fixa), para não expor informações reais de clientes (sigilo contábil e LGPD).

**clientes.csv** (60 linhas): id, cnpj, razao_social, uf, setor, porte, regime, honorario_mensal, data_entrada  
**apuracoes.csv** (1440 linhas, 24 meses): id, cliente_id, competencia, faturamento, despesas, imposto, honorario, pago

As duas tabelas se relacionam por `apuracoes.cliente_id = clientes.id` (modelagem relacional também no SQLite via SQLAlchemy).

## 3. Leitura dos dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

clientes = pd.read_csv("../dados/clientes.csv")
apuracoes = pd.read_csv("../dados/apuracoes.csv")
clientes.head()

In [ ]:
apuracoes.head()

In [ ]:
print(clientes.shape, apuracoes.shape)
clientes.info()

## 4. Limpeza e preparação

In [ ]:
print(clientes.isna().sum().sum(), apuracoes.isna().sum().sum())
print(clientes.duplicated().sum(), apuracoes.duplicated().sum())

apuracoes["competencia"] = pd.to_datetime(apuracoes["competencia"] + "-01")
clientes["data_entrada"] = pd.to_datetime(clientes["data_entrada"])

df = apuracoes.merge(clientes.drop(columns=["honorario_mensal"]), left_on="cliente_id", right_on="id", suffixes=("", "_cliente"))
df = df.drop(columns=["id_cliente"])
df.head()

## 5. Engenharia de atributos

In [ ]:
df["carga_tributaria"] = df["imposto"] / df["faturamento"] * 100
df["lucro_bruto"] = df["faturamento"] - df["despesas"]
df["margem_bruta"] = df["lucro_bruto"] / df["faturamento"] * 100
df["ano"] = df["competencia"].dt.year
df["mes"] = df["competencia"].dt.month
df["inadimplente"] = 1 - df["pago"]
df[["faturamento", "imposto", "carga_tributaria", "lucro_bruto", "margem_bruta"]].describe().round(2)

## 6. Análise exploratória

In [ ]:
df.groupby("regime").agg(clientes=("cliente_id", "nunique"), faturamento=("faturamento", "sum"), imposto=("imposto", "sum"), carga_media=("carga_tributaria", "mean")).round(2)

In [ ]:
df.groupby("setor").agg(faturamento=("faturamento", "sum"), honorarios=("honorario", "sum"), inadimplencia=("inadimplente", "mean")).round(3)

In [ ]:
df.groupby("uf").agg(clientes=("cliente_id", "nunique"), faturamento=("faturamento", "sum"), inadimplencia=("inadimplente", "mean")).round(3)

## 7. KPIs

In [ ]:
faturamento_total = df["faturamento"].sum()
imposto_total = df["imposto"].sum()
carga_media = imposto_total / faturamento_total * 100
honorarios_total = df["honorario"].sum()
inadimplencia = df["inadimplente"].mean() * 100

print(f"Clientes: {df['cliente_id'].nunique()}")
print(f"Faturamento apurado: R$ {faturamento_total:,.2f}")
print(f"Impostos apurados: R$ {imposto_total:,.2f}")
print(f"Carga tributária média: {carga_media:.1f}%")
print(f"Receita de honorários: R$ {honorarios_total:,.2f}")
print(f"Inadimplência: {inadimplencia:.1f}%")

## 8. Gráficos

In [ ]:
mensal = df.groupby("competencia")[["faturamento", "imposto"]].sum().reset_index()
plt.figure(figsize=(10, 4))
sns.lineplot(data=mensal, x="competencia", y="faturamento", label="Faturamento")
sns.lineplot(data=mensal, x="competencia", y="imposto", label="Impostos")
plt.title("Faturamento e impostos por mês")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="regime", y="carga_tributaria", hue="regime", legend=False)
plt.title("Carga tributária por regime")
plt.show()

In [ ]:
por_uf = df.groupby("uf")["faturamento"].sum().reset_index().sort_values("faturamento", ascending=False)
plt.figure(figsize=(8, 4))
sns.barplot(data=por_uf, x="uf", y="faturamento", hue="uf", legend=False)
plt.title("Faturamento por estado")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df[["faturamento", "despesas", "imposto", "honorario", "carga_tributaria", "inadimplente"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlação entre variáveis")
plt.show()

## 9. Interpretação dos resultados
- A carga tributária varia bastante entre regimes; o Lucro Real aparece com a maior carga média na base, porque concentra clientes maiores.
- O faturamento tem sazonalidade e tendência de crescimento ao longo dos 24 meses.
- Faturamento, imposto e honorário são fortemente correlacionados (clientes maiores pagam mais), mas a inadimplência quase não se relaciona com eles.
- A inadimplência de honorários fica em torno de 9% e sobe nos meses mais recentes.

## 10. Conclusão
O projeto mostra como a análise de dados apoia a gestão de um escritório contábil: permite comparar regimes, acompanhar a evolução do faturamento e identificar riscos de inadimplência. Como próximos passos, o dashboard Streamlit adiciona filtros, simulador tributário, consulta de CNPJ por API e banco SQLite.

*Observação: base fictícia e cálculos tributários simplificados, apenas para fins educacionais.*